# Lab 11: Structured Text Extraction

**Business question:** Can complaint text become a validated table with traceable evidence?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | The 23 customer messages with text and 20 reviewed complaints | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | 60 reviewed CFPB complaints | about 75–90 min (Exercise 2 means reading complaints closely) | Three exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

Keep the **Lab Companion** for Module 11 open beside this notebook. It explains each step and shows the output you should expect.

**Extraction** turns free text into fields in a table: an amount, a request, a date. You compare a **rule baseline** (patterns written by hand) with two versions of an LLM prompt. You do not call an LLM. I ran both prompts once and saved every answer in `data/llm_outputs`, so everyone checks the same outputs. A file of saved model answers like this is called a **cache**.

## Part 1 Guided core

### 1.1 Import the libraries

Select the `ban6403` kernel in VS Code, then run the cell below.

In [ ]:
# Path builds file paths that work on Windows, Mac and Linux.
from pathlib import Path
# json reads saved model answers; re runs regular expressions (text patterns).
import json
import re
import numpy as np
import pandas as pd

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the course data is one folder up, in `data`. Your saved files go to `outputs/m11`.

In [ ]:
# ".." means one folder up; the / operator joins folder and file names.
DATA = Path("../data")
OUTPUT_DIR = Path("../outputs/m11")
# parents=True also creates the outputs folder if needed; exist_ok=True means no error if the folder exists.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 A small schema for the customer messages

A **schema** is the list of fields you want, with the allowed values and a rule for missing values. Before any extraction, write it down. Here is a small schema for the 23 customer messages from Module 1 that have text:

| Field | Allowed values | Missing value |
|---|---|---|
| `aspect` | accessories, delivery, billing_refund, device_performance, support_service, appearance | `none` |
| `request` | refund, send_item, help, return_item | `none` |
| `aspect_evidence` | the word or words that triggered the aspect rule, copied from the message | `""` |
| `request_evidence` | the word or words that triggered the request rule, copied from the message | `""` |

A request counts only when the customer asks for something. "The refund arrived yesterday" is not a request.

In [ ]:
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
# Drop the message with no text; reset_index numbers the rows 0, 1, 2, ... again.
feedback = feedback.dropna(subset=["text"]).reset_index(drop=True)
print("messages:", len(feedback))

### 1.4 Extract aspect and request with supplied rules

Each rule is a **regular expression** (a text pattern). The aspect rules are tried in order, and the first pattern found wins. A request rule fires only in a sentence that also contains an asking word such as *please*, *can*, *could* or *need*. The function returns the label and the matched words as evidence.

You need to understand what each supplied pattern is for, not write patterns like these yourself. A short key to the symbols in this lab:

| Symbol | Meaning | Example |
|---|---|---|
| `r"..."` | a raw string: Python keeps each backslash as written | `r"\s+"` |
| `a\|b` | `a` or `b` | `refund\|charged` |
| `\s` / `\d` | one space character / one digit | `\d{4}` = four digits |
| `+`, `?`, `{2,}` | one or more / zero or one (optional) / two or more of the thing before | `months?` = month or months |
| `[...]` | any one of the characters inside | `[.!?]` = a period, `!` or `?` |
| `(?:...)` | groups a part without saving it | `(?:please\|can)` |
| `(...)` | groups a part and saves it as a group you can read later | `\{\$([\d,]+\.\d{2})\}` saves the number |
| `\b` | the edge of a word | `\bcan\b` does not match "scan" |
| `(?<=[.!?])\s+` | spaces that come right after `.`, `!` or `?`: where one sentence ends | used to split sentences |

In [ ]:
# Each rule is a (label, pattern) pair. In a pattern, | means "or".
ASPECT_RULES = [("accessories", r"charger|cable|cushion|case"),
                ("delivery", r"deliver|package|arrived"),
                ("billing_refund", r"refund|charged|bill"),
                ("device_performance", r"battery|sound|start|stopped working|screen|microphone|failed"),
                ("support_service", r"support|service|help"),
                ("appearance", r"color|photo")]
REQUEST_RULES = [("refund", r"refund"), ("send_item", r"send"),
                 ("return_item", r"return"), ("help", r"help|explain|troubleshoot")]
# \b marks the edge of a word, so "can" does not match inside "scan".
ASKING = r"\b(?:please|can|could|need)\b"

def first_rule(text, rules):
    """Take a text and a list of (label, pattern) rules; return the first label found and the matched words."""
    for label, pattern in rules:
        # re.search looks for the pattern anywhere in the text; IGNORECASE ignores upper and lower case.
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            # match.group(0) is the piece of text that matched.
            return label, match.group(0)
    return "none", ""

def extract_feedback(text):
    """Take one message; return a dictionary with the aspect, the request and the evidence for each."""
    aspect, aspect_evidence = first_rule(text, ASPECT_RULES)
    # Split the message into sentences (after . ! or ?) and keep the ones with an asking word.
    asking = []
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if re.search(ASKING, sentence, flags=re.IGNORECASE):
            asking.append(sentence)
    request, request_evidence = first_rule(" ".join(asking), REQUEST_RULES)
    return {"aspect": aspect, "aspect_evidence": aspect_evidence,
            "request": request, "request_evidence": request_evidence}

extract_feedback("The headset stopped working. Please arrange a refund.")

### 1.5 Run the rules on all customer messages

Look back at 1.4. The aspect came out as `billing_refund`, although the headset stopped working, because the billing pattern comes before the device pattern in the list. The order of the rules is part of the rules.

Each output row keeps the `feedback_id`, so every value links back to its message.

In [ ]:
# Run the rules on each message; each output row starts with the message ID.
# zip walks through two columns side by side, giving one (ID, text) pair per message.
rule_rows = []
for feedback_id, text in zip(feedback["feedback_id"], feedback["text"]):
    row = {"feedback_id": feedback_id}
    row.update(extract_feedback(text))  # update adds the keys of the rule output to row
    rule_rows.append(row)
# Put the message text next to the rule output, matched by feedback_id.
message_rules = pd.DataFrame(rule_rows).merge(feedback[["feedback_id", "text"]], on="feedback_id")
print(message_rules["aspect"].value_counts().to_string())
print()
print(message_rules["request"].value_counts().to_string())

### 1.6 Read where the rules go wrong

Rules are fast and easy to explain, but they only see their patterns. Read these four rows. Which labels would a person change?

In [ ]:
# isin keeps the rows whose ID is in the list; the second list picks the columns.
message_rules.loc[message_rules["feedback_id"].isin(["FB005", "FB008", "FB015", "FB019"]),
             ["feedback_id", "text", "aspect", "aspect_evidence", "request", "request_evidence"]]

There are three kinds of mistakes here:

- A **wrong value** fills the field with the wrong label. FB005 is about a refund, but the word "arrived" makes the rule say `delivery`. FB015 asks for troubleshooting steps, but "send" comes first in the rule order, so the request becomes `send_item`.
- A **missing extraction** leaves a field empty that should be filled. FB008 asks the shop to check the bill, but no request pattern covers "check".
- FB019 is different: it is about comfort, and comfort is not one of the allowed aspects. This is a **gap in the schema**, not a missed rule. The fix is to decide whether the schema needs a `comfort` aspect.
- A **false extraction** fills a field with something that is not there. None of the customer messages trigger one, so here is a made-up sentence that does: the customer says they do *not* need a refund.

In [ ]:
extract_feedback("I do not need a refund. The speaker works fine.")  # made-up example

### 1.7 The complaint extraction schema

Now the real task. A complaint team wants one table row per CFPB complaint with four fields. Read the full schema in `data/llm_outputs/prompts/m11_extraction_schema.md` before you go on. Its key rules:

| Field | Missing value | Evidence |
|---|---|---|
| `disputed_amount`: the main dollar amount in dispute | `null` | `amount_evidence` |
| `requested_remedy`: what the consumer explicitly asks for (7 labels) | `not_stated` | `remedy_evidence` |
| `time_reference`: the first date or length of time about the problem | `null` | the value itself |
| `contacted_company`: did the consumer contact a business involved? | `not_stated` | `contact_evidence` |

Every evidence field must be copied exactly from the complaint. The cell loads the JSON schema that the LLM had to follow.

In [ ]:
# json.loads turns the JSON text into a Python dictionary.
schema = json.loads((DATA / "llm_outputs/prompts/m11_extract_schema.json").read_text())
# One line per key: its name, its type, "or null" when it may be missing, and its allowed labels.
for name, spec in schema["properties"].items():
    # spec.get(key, default) returns the default when the key is not there.
    nullable_text = "or null" if spec.get("nullable") else ""
    # {name:>18} right-aligns the name in 18 characters, so the colons line up.
    print(f"{name:>18}: {spec['type']}", nullable_text, spec.get("enum", ""))

### 1.8 Load the 20-complaint lab sample

The reviewed file holds 60 complaints from the `valid` split. Their labels are draft judgments, made by reading each complaint against the schema. They are careful readings, not ground truth. In Part 1 you use the 20 marked `lab_sample`. In the file, an empty cell means a missing value, so we read it with `keep_default_na=False`: empty text stays an empty string. The amount column is then turned into numbers.

In [ ]:
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
# keep_default_na=False: an empty cell stays an empty string ("") instead of becoming NaN.
reviewed = pd.read_csv(DATA / "reviewed/m11_reviewed_dev.csv", keep_default_na=False)
# errors="coerce" turns the empty amounts into NaN (missing) and the others into numbers.
reviewed["disputed_amount"] = pd.to_numeric(reviewed["disputed_amount"], errors="coerce")
# Add the complaint text from the course file, matched by complaint_id.
reviewed = reviewed.merge(complaints[["complaint_id", "narrative"]], on="complaint_id")
# lab_sample is True or False, so it can select the rows directly.
lab = reviewed[reviewed["lab_sample"]].reset_index(drop=True)
print("reviewed:", len(reviewed), "| lab sample:", len(lab))
lab[["complaint_id", "product", "disputed_amount", "requested_remedy", "time_reference", "contacted_company"]].head(6)

### 1.9 The rule baseline for complaints

These rules follow the schema as closely as simple patterns can:

- **amount**: the first `{$...}` in the text, with up to three words before it as evidence;
- **time**: the first redacted date, year, or phrase such as "3 days" or "two months ago";
- **contact**: a sentence part that starts with "I called", "I contacted", "I disputed" and similar;
- **remedy**: in sentences with an asking word, the first label whose words appear, in the schema's precedence order.

The rules were written for this course after reading complaints like these, so they may fit this sample better than new complaints.

In [ ]:
# Amount: up to three words, then {$1,234.56}. The part in plain ( ) is "captured" as group 1: the number.
AMOUNT = r"(?:\S+\s+){0,3}\{\$([\d,]+\.\d{2})\}"
# Time: a redacted date, a year from 1900 to 2099, or a phrase such as "3 days" or "two months ago".
TIME = (r"XX/XX/(?:XXXX|\d{4}|year>|scrub>)(?:,? \d{4})?|\b(?:19|20)\d{2}\b"
        r"|\b(?:\d+|XXXX|a|one|two|three|four|five|six|several|few)\s+(?:days?|weeks?|months?|years?)(?:\s+ago)?\b")
# Contact: "I called", "We contacted", "I have disputed" and similar, up to the end of the sentence.
CONTACT = (r"\b(?:I|We|i)\s+(?:have\s+|had\s+|also\s+|then\s+|immediately\s+)?"
           r"(?:called|contacted|emailed|spoke|wrote|disputed|filed a dispute|sent (?:them|a letter)|asked)\b[^.!?\n]*")
# Words that show the consumer asks for something.
CUE = r"\b(?:request|requesting|ask|asking|want|need|demand|please|should|seeking|expect)\b"
# Remedy labels in the schema's precedence order: the first one found wins.
REMEDY_RULES = [("refund", r"refund|reimburs|revers|money back|restor|recover"),
                ("correct_report", r"delet|remov|correct|block"),
                ("stop_or_validate_debt", r"validat|proof|cease|stop"),
                ("account_action", r"access|unfreeze|unlock|reopen|release"),
                ("review_or_explain", r"investigat|review|explain|document|verif")]

In [ ]:
def rule_extract(text):
    """Take one complaint; return a dictionary with the four fields and their evidence."""
    row = {}
    # Amount: group(1) is the number inside {$...}; remove the commas and make it a float.
    m = re.search(AMOUNT, text)
    if m:
        row["disputed_amount"] = float(m.group(1).replace(",", ""))
        row["amount_evidence"] = m.group(0)
    else:
        row["disputed_amount"] = np.nan
        row["amount_evidence"] = ""

    # Remedy: keep the sentences (or lines) with an asking word, then apply the remedy rules to them.
    asking = []
    for sentence in re.split(r"(?<=[.!?])\s+|\n+", text):
        if re.search(CUE, sentence, flags=re.IGNORECASE):
            asking.append(sentence)
    remedy, matched_words = first_rule(" ".join(asking), REMEDY_RULES)
    if remedy == "none":
        row["requested_remedy"] = "not_stated"
        row["remedy_evidence"] = ""
    else:
        row["requested_remedy"] = remedy
        # Keep the whole asking sentence that contains the matched words as evidence.
        for sentence in asking:
            if re.search(matched_words, sentence, flags=re.IGNORECASE):
                row["remedy_evidence"] = sentence
                break

    # Time and contact: the first match, or the missing value when there is none.
    m = re.search(TIME, text)
    row["time_reference"] = m.group(0) if m else ""
    m = re.search(CONTACT, text)
    row["contacted_company"] = "yes" if m else "not_stated"
    row["contact_evidence"] = m.group(0) if m else ""
    return row

# Run the rules on the 20 lab complaints; each row starts with the complaint ID.
rule_rows = []
for complaint_id, text in zip(lab["complaint_id"], lab["narrative"]):
    row = {"complaint_id": complaint_id}
    row.update(rule_extract(text))
    rule_rows.append(row)
rules_lab = pd.DataFrame(rule_rows)
rules_lab[["complaint_id", "disputed_amount", "requested_remedy", "time_reference", "contacted_company"]].head(6)

### 1.10 Optional: try a revised prompt yourself (not graded)

You may try the extraction by hand on three complaints from the public course data: **16582909**, **15877499** and **15196313**. Use Google AI Studio (free with a Google account) or UTSA Copilot Chat. Do not paste anything other than these public complaints.

1. Open `data/llm_outputs/prompts/m11_extract_v1_prompt.md`. Add one rule that you think is missing, for example the missing-value rule for `requested_remedy`.
2. Paste your revised prompt with one complaint in place of `{narrative}`. Repeat for the other two.
3. Compare your answers with the captured answers printed below, which come from my run of prompt version 2.

If you skip this step or cannot get access, the cell below is the captured equivalent. Chat answers can differ from run to run, so treat your trial as an illustration, not a measurement.

In [ ]:
TRIAL_IDS = [16582909, 15877499, 15196313]
# A .jsonl file has one JSON record per line; json.loads reads each line into a dictionary.
v2_records = []
for line in (DATA / "llm_outputs/m11_extract_v2_gemini.jsonl").read_text().splitlines():
    v2_records.append(json.loads(line))

for record in v2_records:
    if int(record["id"]) in TRIAL_IDS:
        print(record["id"], "|", record["prompt_version"], "|", record["model_version"])
        # The complaint text from the lab sample: .iloc[0] takes the one matching row.
        narrative = lab.loc[lab["complaint_id"].eq(int(record["id"])), "narrative"].iloc[0]
        print("  ", narrative[:160], "...")
        # "parsed" holds the model's answer as a dictionary.
        print("  ", record["parsed"])

### 1.11 Load the two-version cache by complaint ID

I sent the same 60 complaints to gemini-3.5-flash-lite with two prompts:

- `m11_extract_v1`: the field names and label lists only;
- `m11_extract_v2`: the full task specification from the schema, a copying rule and three made-up examples.

Both used the same JSON schema. Each record also stores the model version, the date, the time taken and the token counts. We join every output to the reviewed labels by `complaint_id`, never by row order.

**Schema versions.** Both prompts were written from schema version 1. The reviewed labels follow version 1.2, the schema file you read in 1.7, which added a few rules after the prompts were written. So the scores in this lab compare the saved outputs against version 1.2, and some disagreements come from changed instructions, not from the model. Step 2.3 names the complaints this affects. The provenance record in 1.15 lists both versions.

Three terms you will see in the records:

- **JSON** is a plain-text format of named keys and values, for example `{"disputed_amount": 900, "requested_remedy": "refund"}`. Python reads it into a dictionary with `json.loads`.
- A **JSON schema** is a written list of the keys an answer must have, with their types and allowed labels. Here it requires seven keys: the four fields and three evidence keys.
- **Temperature** controls how much randomness the model uses when it picks its next words. At temperature 0 it picks the most likely words, so a rerun usually gives the same answer, but not always. That is one reason the outputs are cached.

In [ ]:
def load_cache(path):
    """Take a .jsonl file of saved answers; return the list of records and a table with one row per complaint."""
    records = []
    for line in path.read_text().splitlines():
        records.append(json.loads(line))
    # One row per answer: the complaint ID, then the fields of the parsed answer.
    rows = []
    for record in records:
        row = {"complaint_id": int(record["id"])}
        row.update(record["parsed"])
        rows.append(row)
    out = pd.DataFrame(rows)
    # A missing amount is null in JSON; to_numeric gives a number column with NaN for missing.
    out["disputed_amount"] = pd.to_numeric(out["disputed_amount"])
    # A missing time reference becomes empty text, the schema's missing value.
    out["time_reference"] = out["time_reference"].fillna("")
    return records, out

v1_records, v1 = load_cache(DATA / "llm_outputs/m11_extract_v1_gemini.jsonl")
v2_records, v2 = load_cache(DATA / "llm_outputs/m11_extract_v2_gemini.jsonl")

# A short summary of each file: errors, prompt, model, date and input tokens.
for name, records in [("v1", v1_records), ("v2", v2_records)]:
    n_errors = 0
    input_tokens = 0
    # A set keeps each value once: .add puts in a model name, and repeats are ignored.
    models = set()
    for record in records:
        if record["error"] is not None:
            n_errors += 1
        input_tokens += record["input_tokens"]
        models.add(record["model_version"])
    # created[:10] keeps the date part (YYYY-MM-DD) of the time stamp.
    print(name, "| records:", len(records), "| errors:", n_errors,
          "| prompt:", records[0]["prompt_version"], "| model:", sorted(models),
          "| date:", records[0]["created"][:10], "| input tokens:", input_tokens)

# Keep the outputs for the 20 lab complaints, matched by complaint_id (never by row order).
v1_lab = lab[["complaint_id"]].merge(v1, on="complaint_id")
v2_lab = lab[["complaint_id"]].merge(v2, on="complaint_id")

### 1.12 Schema validity

An output is **valid** when it follows the schema's form. The JSON response schema already forced the model to return all seven keys with the right types. The helper checks the rest of the schema's rules. Labels must be allowed, and text fields must be text. The amount must match its evidence. A `not_stated` label goes with empty evidence, and the reverse. Every evidence text must appear exactly in the complaint. The helper returns a list of problems; an empty list means valid.

Validity says nothing about whether the values are right. That comes next.

In [ ]:
# The allowed labels, read from the schema.
REMEDIES = schema["properties"]["requested_remedy"]["enum"]
CONTACTS = schema["properties"]["contacted_company"]["enum"]
TEXT_FIELDS = ["amount_evidence", "remedy_evidence", "contact_evidence", "time_reference"]
# A dollar amount as written in the complaints, such as {$1,234.56}; group 1 is the number.
AMOUNT_IN_TEXT = r"\{\$([\d,]+\.\d{2})\}"

def validity_problems(row, text):
    """Take one output row and its complaint text; return a list of problems (an empty list means valid)."""
    problems = []
    if row["requested_remedy"] not in REMEDIES or row["contacted_company"] not in CONTACTS:
        problems.append("label not allowed")
    # The checks below need text, so stop here if a text field is not a string.
    for col in TEXT_FIELDS:
        if not isinstance(row[col], str):
            problems.append("text field is not text")
            return problems

    # A missing amount (NaN) goes with empty evidence, and an amount with some evidence.
    amount_missing = pd.isna(row["disputed_amount"])
    evidence_empty = row["amount_evidence"] == ""
    if amount_missing != evidence_empty:
        problems.append("amount and evidence disagree")
    elif not amount_missing:
        # Read every {$...} amount in the evidence as a number (commas removed),
        # then check that one of them equals the extracted amount, to the cent.
        found = False
        for number in re.findall(AMOUNT_IN_TEXT, row["amount_evidence"]):
            if abs(float(number.replace(",", "")) - row["disputed_amount"]) < 0.005:
                found = True
        if not found:
            problems.append("amount not in its evidence")

    # not_stated goes with empty evidence, and a label with some evidence.
    if (row["requested_remedy"] == "not_stated") != (row["remedy_evidence"] == ""):
        problems.append("remedy and evidence disagree")
    if (row["contacted_company"] == "not_stated") != (row["contact_evidence"] == ""):
        problems.append("contact and evidence disagree")

    # Every evidence text must be copied exactly from the complaint.
    for col in TEXT_FIELDS:
        if row[col] and row[col] not in text:
            problems.append(f"{col} not in text")
    return problems

def check_validity(out, source):
    """Take an output table and the complaints; return one list of problems per output row."""
    # A dictionary from complaint ID to complaint text.
    texts = dict(zip(source["complaint_id"], source["narrative"]))
    all_problems = []
    # iterrows gives the rows one at a time, as (index, row) pairs.
    for _, row in out.iterrows():
        all_problems.append(validity_problems(row, texts[row["complaint_id"]]))
    return all_problems

for name, out in [("rules", rules_lab), ("v1", v1_lab), ("v2", v2_lab)]:
    problems = check_validity(out, lab)
    n_valid = 0
    for row_problems in problems:
        if len(row_problems) == 0:
            n_valid += 1
    print(f"{name:>5}: valid rows {n_valid} of {len(out)}")

Look at the v1 problems. Most come from evidence that is not copied exactly, or from evidence given for a missing amount.

In [ ]:
v1_lab["problems"] = check_validity(v1_lab, lab)
# .str.len() gives the length of each list; > 0 keeps the rows with at least one problem.
v1_lab.loc[v1_lab["problems"].str.len() > 0, ["complaint_id", "disputed_amount", "amount_evidence", "problems"]]

### 1.13 Field correctness against the reviewed labels

A value is **correct** when it matches the reviewed label: the same number for amounts and the same label for remedy and contact. For time, the two texts must be equal after a small cleanup: lower case, single spaces, and leading context words such as *on*, *since* or a weekday removed. So "On XX/XX/2025" matches "XX/XX/2025", but "3 days" does not match "13 days". Two missing values also match. The helper `same_value` applies these rules, and `error_type` names the kind of mistake.

Every scoring rule is a choice. A looser rule, such as "one text contains the other", would count more values as correct and make every method look better.

In [ ]:
FIELDS = ["disputed_amount", "requested_remedy", "time_reference", "contacted_company"]
MISSING = {"requested_remedy": "not_stated", "time_reference": "", "contacted_company": "not_stated"}
# Context words that may come before a time value without changing it ("on XX/XX/2025" = "XX/XX/2025").
TIME_LEAD_WORDS = ["on", "in", "since", "for", "today",
                   "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

def is_missing(field, value):
    """Take a field name and a value; return True when the value is that field's missing value."""
    if field == "disputed_amount":
        return pd.isna(value)
    return value == MISSING[field]

def normal_time(text):
    """Take a time value; return it in lower case, with single spaces, without leading context words or end commas and periods."""
    words = text.lower().split()
    while len(words) > 0 and words[0] in TIME_LEAD_WORDS:
        words = words[1:]
    return " ".join(words).strip(" .,")

def same_value(field, out, ref):
    """Take a field, an output value and a reviewed value; return True when they count as the same."""
    # Two missing values match; a missing value and a filled one do not.
    if is_missing(field, out) or is_missing(field, ref):
        return is_missing(field, out) and is_missing(field, ref)
    # Amounts: equal to the cent.
    if field == "disputed_amount":
        return abs(out - ref) < 0.005
    # Time: equal after normalizing ("3 days" does not match "13 days").
    if field == "time_reference":
        return normal_time(out) == normal_time(ref)
    return out == ref

def error_type(field, out, ref):
    """Take a field, an output value and a reviewed value; return "correct" or the kind of error."""
    if same_value(field, out, ref):
        return "correct"
    if is_missing(field, ref):
        return "false extraction"
    if is_missing(field, out):
        return "missing extraction"
    return "wrong value"

def correct_counts(out, ref):
    """Take an output table and the reviewed table; return the number of correct values for each field."""
    # Join by complaint_id; the output's columns get the ending _out.
    both = ref.merge(out, on="complaint_id", suffixes=("", "_out"))
    counts = {}
    for f in FIELDS:
        n_correct = 0
        for out_value, ref_value in zip(both[f + "_out"], both[f]):
            if same_value(f, out_value, ref_value):
                n_correct += 1
        counts[f] = n_correct
    return counts

# One column per method, one row per field.
lab_table = pd.DataFrame({"rules": correct_counts(rules_lab, lab),
                          "v1": correct_counts(v1_lab, lab),
                          "v2": correct_counts(v2_lab, lab)})
lab_table["out_of"] = len(lab)
lab_table

### 1.14 False and missing extractions for the remedy field

A single correct count hides the kind of error. A **false extraction** fills a field the reviewer left missing; a **missing extraction** leaves out something the reviewer found; a **wrong value** fills it with a different value. These errors have different costs: a false refund request wastes an agent's time, a missing one leaves a customer waiting.

In [ ]:
remedy_errors = {}
for name, out in [("rules", rules_lab), ("v1", v1_lab), ("v2", v2_lab)]:
    both = lab.merge(out, on="complaint_id", suffixes=("", "_out"))
    # Name the kind of error for each complaint, then count each kind.
    kinds = []
    for out_value, ref_value in zip(both["requested_remedy_out"], both["requested_remedy"]):
        kinds.append(error_type("requested_remedy", out_value, ref_value))
    remedy_errors[name] = pd.Series(kinds).value_counts()
# A kind of error that a method never made is NaN in the table; fillna(0) makes it 0.
pd.DataFrame(remedy_errors).fillna(0).astype(int)

### 1.15 Which outputs are real? A provenance record

The prompt for v2 contains three answers in exactly the same JSON form as the model's answers. But I wrote those as examples. They are not model outputs. A file that mixes the two would mislead a reader.

A record of where each output came from is called **provenance**. The first cell shows one written example from the prompt. The second cell builds a provenance record for every output you used in this lab. It says what kind each output is, which prompt, schema version and model version made it, and when.

In [ ]:
prompt_v2 = (DATA / "llm_outputs/prompts/m11_extract_v2_prompt.md").read_text()
# Find each "Answer: {{...}}" in the prompt. The prompt doubles the braces, so turn {{ }} back into { }.
written_examples = []
for answer in re.findall(r"Answer: (\{\{.*\}\})", prompt_v2):
    written_examples.append(json.loads(answer.replace("{{", "{").replace("}}", "}")))
print("examples written into prompt v2:", len(written_examples))
print(written_examples[2])

In [ ]:
def cache_entry(file, records):
    """Take a file name and its saved records; return a provenance entry (a dictionary) for the file."""
    created = []
    models = set()
    n_errors = 0
    for record in records:
        created.append(record["created"])
        models.add(record["model_version"])
        if record["error"] is not None:
            n_errors += 1
    return {"file": file, "kind": "real model output", "prompt_version": records[0]["prompt_version"],
            "schema_version": "prompt written from schema version 1",
            "model_version": sorted(models), "temperature": records[0]["temperature"],
            "created": [min(created), max(created)], "rows": len(records), "errors": n_errors}

# One entry per source of outputs used in this lab.
provenance = [
    cache_entry("data/llm_outputs/m11_extract_v1_gemini.jsonl", v1_records),
    cache_entry("data/llm_outputs/m11_extract_v2_gemini.jsonl", v2_records),
    {"file": "data/llm_outputs/prompts/m11_extract_v2_prompt.md (Examples A-C)", "kind": "examples written into the prompt (made up)",
     "schema_version": "schema version 1", "rows": len(written_examples)},
    {"file": "rule baseline in this notebook (step 1.9)", "kind": "rule output", "schema_version": "hand-written patterns",
     "rows": len(rules_lab)},
    {"file": "data/reviewed/m11_reviewed_dev.csv", "kind": "reviewed labels (draft judgments)", "schema_version": "schema version 1.2",
     "rows": len(reviewed)},
]
pd.DataFrame(provenance)[["file", "kind", "schema_version", "rows"]]

### 1.16 Save your outputs

This writes the rule outputs, the correctness table and the provenance record to `outputs/m11`. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
message_rules.to_csv(OUTPUT_DIR / "feedback_rules.csv", index=False)
rules_lab.to_csv(OUTPUT_DIR / "lab_rules.csv", index=False)
lab_table.to_csv(OUTPUT_DIR / "lab_correctness.csv")
# json.dumps turns the list of dictionaries into JSON text; indent=2 makes it easy to read.
(OUTPUT_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2))
print("saved in outputs/m11: feedback_rules.csv, lab_rules.csv, lab_correctness.csv, provenance.json")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. In step 1.12, prompt v1 had fewer valid rows than v2. Does a valid row mean the values are correct? Use one number from step 1.13 or 1.14.
2. Why does the provenance record list the three examples in prompt v2 separately from the model outputs?

**Your answer:**

## Part 2 Practice: the full reviewed sample

Now use all 60 reviewed complaints: 10 per product, half of them with a dollar amount. These are draft judgments. Read the complaint and compare your reasoning with the `review_note` column, which explains the hard cases. Keep the schema file open while you work.

### 2.1 Run the rule baseline on all 60 complaints

In [ ]:
# Same rules as in 1.9, now on all 60 reviewed complaints.
rule_rows = []
for complaint_id, text in zip(reviewed["complaint_id"], reviewed["narrative"]):
    row = {"complaint_id": complaint_id}
    row.update(rule_extract(text))
    rule_rows.append(row)
rules_all = pd.DataFrame(rule_rows)
print("rule rows:", len(rules_all), "| v1 rows:", len(v1), "| v2 rows:", len(v2))

### 2.2 Schema validity on all 60

Count valid rows for each method, then list the kinds of problems.

In [ ]:
outputs = {"rules": rules_all, "v1": v1, "v2": v2}
for name, out in outputs.items():
    # Add a column with the list of problems of each row.
    out["problems"] = check_validity(out, reviewed)
    n_valid = (out["problems"].str.len() == 0).sum()
    print(f"{name:>5}: valid rows {n_valid} of {len(out)}")
print()
print("v1 problem counts:")
# explode puts each problem of each list on its own row, so value_counts can count them.
print(v1["problems"].explode().value_counts().to_string())

### 2.3 What the model did with missing remedies

For some complaints the reviewed label is `not_stated`: the consumer describes the problem but asks for nothing specific. Compare what each method answered for those complaints.

**A note on the reference labels.** As step 1.11 said, the reviewed labels follow schema version 1.2. Version 1.1 added a rule on requests addressed to the CFPB. Version 1.2 added, among others, a rule that a dispute, appeal or application the consumer reports making earlier counts as a request. Prompt v2 was written from version 1 and never saw these rules. So some of v2's remedy errors come partly from the rule changes, not only from the model. One is the false extraction for complaint 14896403 (version 1.1). Four are missing requests: complaints 14415194, 16824177, 14600829 and 16774146 (version 1.2). Their `review_note` names the schema version.

In [ ]:
no_request = reviewed.loc[reviewed["requested_remedy"].eq("not_stated"), ["complaint_id"]]
print("complaints reviewed as not_stated:", len(no_request))
# For each method, count the remedy labels it gave to these complaints.
answers = {}
for name, out in outputs.items():
    method_labels = no_request.merge(out, on="complaint_id")["requested_remedy"]
    answers[name] = method_labels.value_counts()
pd.DataFrame(answers).fillna(0).astype(int)

### Exercise 1: field correctness for three methods

Use `correct_counts` from step 1.13 on all 60 reviewed complaints. Make one table with a row per field (`disputed_amount`, `requested_remedy`, `time_reference`, `contacted_company`) and a column per method (`rules`, `v1`, `v2`). Add an `out_of` column. Then add a row `valid_rows` with each method's number of valid rows from step 2.2.

Hint: build a dictionary with one entry per method, then turn it into a table:

```python
counts = {}
for name, out in outputs.items():
    counts[name] = correct_counts(out, reviewed)
table = pd.DataFrame(counts)
```

In [ ]:
# Exercise 1: your code here
# TODO: build a table with one row per field and one column per method (rules, v1, v2), using correct_counts.
# TODO: add the column out_of (60).
# TODO: add a row valid_rows with each method's number of valid rows from step 2.2.

### Exercise 2: false, missing and wrong extractions

1. For each of the four fields, count the error types (`correct`, `false extraction`, `missing extraction`, `wrong value`) for the **rules** and for **v2**, using `error_type` as in step 1.14. Show one small table per field, or one combined table.
2. List the complaints where v2's `requested_remedy` differs from the reviewed label, with `complaint_id`, the reviewed label, v2's label and v2's `remedy_evidence`.
3. Print the full narrative and the reviewed `review_note` of complaints 14896403 and 18146055. Both are in your list, and both have a note. Decide whether you agree with the reviewer, with v2, or with neither. (Many other disagreements have no note; you may add one of those and reason from the schema.)

In [ ]:
# Exercise 2: your code here
# TODO: for each field, count the error types of the rules and of v2 with error_type (as in step 1.14).
# TODO: list the complaints where v2's requested_remedy differs from the reviewed label
#       (complaint_id, reviewed label, v2 label, v2 remedy_evidence).
# TODO: print the narrative and review_note of complaints 14896403 and 18146055; write your decision in a comment.

### Exercise 3: a validated extraction table with provenance

Build the table a complaint team would receive from the v2 outputs:

1. One row per complaint. Keep `complaint_id` and the seven v2 keys: `disputed_amount`, `amount_evidence`, `requested_remedy`, `remedy_evidence`, `time_reference`, `contacted_company`, `contact_evidence`. Add a column `needs_review` that is True when the row has any validity problem. Leave out the `problems` column from step 2.2.
2. Count the rows that need review.
3. Save the table as `outputs/m11/validated_extraction_v2.csv` and add an entry for it to `provenance` (file, kind, prompt version, model version, rows). Save `provenance.json` again.

Hint: start from `v2.drop(columns="problems").copy()` and use `v2["problems"].str.len() > 0`.

In [ ]:
# Exercise 3: your code here
# TODO: start from v2.drop(columns="problems").copy(); add needs_review (True when the row has any validity problem).
# TODO: print how many rows need review.
# TODO: save the table as validated_extraction_v2.csv in OUTPUT_DIR, add a provenance entry and save provenance.json again.

### Practice interpretation

Write 3–5 sentences:

- Using Exercise 1, which method would you trust for each field? Give the counts out of 60.
- Using step 2.3 and Exercise 2, which kind of error does each method make most for `requested_remedy`, and which kind would cost the complaint team more?
- Using Exercise 3, is "valid" enough to send the table to the team? What would you add, and what does the provenance record let a reader check?

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

Did you use an AI tool to explain or debug code, or for the optional trial in step 1.10? Then add one line here. Name the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Make one rule follow the schema more closely

The schema says time spent as a customer does not count as a time reference ("I have been a customer for 20 years"). The time rule ignores this. Write a version that skips matches preceded by words such as *customer for* or *member for*. Then count correct `time_reference` values again for all 60 complaints.

In [ ]:
# Challenge A: your code here
# TODO: write a time rule that skips matches after words such as "customer for" or "member for".
# TODO: run it on all 60 complaints and count correct time_reference values with same_value.

### B. Pretrained named-entity recognition

A **named-entity recognizer** (NER) is a pretrained model that tags spans such as MONEY and DATE. It does not know our schema: it finds every amount, not the disputed one. The `spacy` package and its small English model are not in the course environment. If you install them (`pip install spacy`, then `python -m spacy download en_core_web_sm`), this cell tags one complaint.

In [ ]:
# spaCy is optional: if it is not installed, the cell prints a note instead of stopping Run All.
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except (ImportError, OSError):
    print("spaCy or its English model is not installed. This part is optional.")
else:
    doc = nlp(reviewed.loc[reviewed["complaint_id"].eq(15812976), "narrative"].iloc[0])
    # Keep the money and date entities that spaCy found.
    entities = []
    for ent in doc.ents:
        if ent.label_ in ("MONEY", "DATE"):
            entities.append((ent.text, ent.label_))
    print(entities)

### C. One prompt, several tasks

Some teams ask one prompt to classify, summarize and extract at once. Draft such a prompt for these complaints: product label, a one-sentence summary, and the four schema fields. What new checks would the summary need, given that it cannot be copied exactly from the text? If you try it in Google AI Studio, use only the three complaints from step 1.10.

## Further reading

- [Jurafsky and Martin, Speech and Language Processing, chapter on information extraction](https://web.stanford.edu/~jurafsky/slp3/)
- [Google AI for Developers: Structured output with the Gemini API](https://ai.google.dev/gemini-api/docs/structured-output)
- [spaCy: Named entities](https://spacy.io/usage/linguistic-features#named-entities)

Next, Module 12 reuses these prompt rules and source IDs to answer support questions from retrieved passages.